In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1" # Required for efficient surrogate evaluation by avoiding multi-threading overheads
import numpy as np
import matplotlib.pyplot as plt
import lal 
import lalsimulation as ls

In [6]:
%%timeit
merger_ringdown_approximant="NRSur7dq4"

mass1 = 7. # masses (in solar masses) 
mass2 = 3.
spin1z = 0.5
spin2z = 0.6
f_lower_mr = 0.
distance = 1.
coa_phase = 0.
# reference_eccentricity = 0.43 # reference eccentricity
# reference_mean_anomaly = 60 * np.pi/180. # reference mean anomaly
# orb_params_list = ['e', 'l', 'x']

distance = 400. # source luminosity distance (in Mpc)
inclination = 30 * np.pi/180. # orbital inclination with line-of-sight

delta_t = 1/2**12 # time grid-spacing (in s)
t_start = -136. # Waveform starting time (in s). t=0 corresponds to the end of waveform, so the starting time should be a negative real number


hlm_mr = ls.SimInspiralChooseTDModes(
    coa_phase,  # phiRef
    delta_t,  # deltaT
    mass1 * lal.MSUN_SI,
    mass2 * lal.MSUN_SI,
    0,  # spin1x
    0,  # spin1y
    spin1z,
    0,  # spin2x
    0,  # spin2y
    spin2z,
    f_lower_mr,  # f_min
    f_lower_mr,  # f_ref
    distance * lal.PC_SI * 1.0e6,
    None,  # LALpars
    4,  # lmax
    getattr(ls, merger_ringdown_approximant),
)

modes_mr = {}
while hlm_mr is not None:
    modes_mr[(hlm_mr.l, hlm_mr.m)] = hlm_mr.mode
    hlm_mr = hlm_mr.next

modes_mr_numpy = {k: np.asarray(modes_mr[k].data.data) for k in modes_mr}

39.9 ms ± 1.5 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [7]:
%%timeit
merger_ringdown_approximant="NRSur7dq4"

mass1 = 7. # masses (in solar masses) 
mass2 = 3.
spin1z = 0.5
spin2z = 0.6
f_lower_mr = 0.
distance = 1.
coa_phase = 0.
# reference_eccentricity = 0.43 # reference eccentricity
# reference_mean_anomaly = 60 * np.pi/180. # reference mean anomaly
# orb_params_list = ['e', 'l', 'x']

distance = 400. # source luminosity distance (in Mpc)
inclination = 30 * np.pi/180. # orbital inclination with line-of-sight

delta_t = 1/2**12 # time grid-spacing (in s)
t_start = -136. # Waveform starting time (in s). t=0 corresponds to the end of waveform, so the starting time should be a negative real number


hlm_mr = ls.SimInspiralChooseTDModes(
    coa_phase,  # phiRef
    delta_t,  # deltaT
    mass1 * lal.MSUN_SI,
    mass2 * lal.MSUN_SI,
    0,  # spin1x
    0,  # spin1y
    spin1z,
    0,  # spin2x
    0,  # spin2y
    spin2z,
    f_lower_mr,  # f_min
    f_lower_mr,  # f_ref
    distance * lal.PC_SI * 1.0e6,
    None,  # LALpars
    4,  # lmax
    getattr(ls, merger_ringdown_approximant),
)

wanted_modes = [(2, 2), (2, -2), (3, 3), (3, -3)]

# modes_mr_numpy = {}
modes_mr_numpy1 = {}
mode = hlm_mr
while mode is not None:
    key = (mode.l, mode.m)
    if key in wanted_modes:
        modes_mr_numpy1[key] = mode.mode.data.data
        # modes_mr_numpy1[key] = np.frombuffer(mode.mode.data.data).copy()
    mode = mode.next

18.9 ms ± 315 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [4]:
# Store a reference value before reassignment
test_key = (2, 2)
test_val_before = modes_mr_numpy1[test_key][0]  # first sample

# Reassign / delete the LAL object
hlm_mr = None  # or let it go out of scope
mode = None
# Check if the numpy array is affected
test_val_after = modes_mr_numpy1[test_key][0]

print(f"Before: {test_val_before}")
print(f"After:  {test_val_after}")
print(f"Same:   {test_val_before == test_val_after}")

print(modes_mr_numpy1[test_key].base)  # None if owns its memory, some object if a view

Before: (1.0158331037658442e-22-2.0418609581234193e-29j)
After:  (1.0158331037658442e-22-2.0418609581234193e-29j)
Same:   True
<Swig Object of type 'tagCOMPLEX16Vector *' at 0x78df97fff7b0>


In [5]:
for key in modes_mr_numpy1.keys():
    print(np.any(modes_mr_numpy1[key] - modes_mr_numpy[key]))

False
False
False
False
